Clip testing

In [4]:
import os
import glob
import random
from typing import Tuple, List, Dict

import numpy as np
from PIL import Image
from tqdm import tqdm
import torch
import open_clip


IMG_EXTS = ("png", "jpg", "jpeg", "webp")


def iter_images(root: str, exts=IMG_EXTS) -> List[str]:
    files = []
    for e in exts:
        files += glob.glob(os.path.join(root, "**", f"*.{e}"), recursive=True)
    return sorted(files)

c:\Anaconda2\envs\lggpt\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

@torch.no_grad()
def embed_images_clip(
    image_paths: List[str],
    model_name: str = "ViT-L-14",
    pretrained: str = "openai",
    batch_size: int = 64,
    normalize: bool = True,
) -> np.ndarray:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
    model = model.to(device).eval()

    all_embs = []

    for i in tqdm(range(0, len(image_paths), batch_size), desc="Embedding"):
        batch_paths = image_paths[i:i + batch_size]
        imgs = []
        for p in batch_paths:
            im = Image.open(p).convert("RGB")
            imgs.append(preprocess(im))
        x = torch.stack(imgs).to(device)

        feats = model.encode_image(x)
        if normalize:
            feats = feats / feats.norm(dim=-1, keepdim=True)

        all_embs.append(feats.cpu().numpy().astype(np.float32))

    embs = np.concatenate(all_embs, axis=0)
    return embs



In [9]:
image_root = "../../../../dataset/traces/test_traces/"
paths = iter_images(image_root)
if not paths:
        raise FileNotFoundError(f"No images found under: {image_root}")

In [18]:
embs2 = embed_images_clip(
    paths,
    model_name="ViT-L-14",
    pretrained="openai",
    batch_size=64,
    normalize=False,
)

c:\Anaconda2\envs\lggpt\lib\site-packages\open_clip\factory.py:388: UserWarning: These pretrained weights were trained with QuickGELU activation but the model config does not have that enabled. Consider using a model config with a "-quickgelu" suffix or enable with a flag.
  warnings.warn(
Embedding: 100%|██████████| 1/1 [00:00<00:00,  4.24it/s]


In [ ]:
embs1=embs

In [19]:
embs1[0][:10]

array([ 0.03481453,  0.07696232,  0.034705  , -0.04101063, -0.00305105,
        0.05513828, -0.00078739,  0.03105987,  0.03248897,  0.01809126],
      dtype=float32)

In [24]:
embs2[0][:10]
print(embs1.max(), embs1.min())
print(embs2.max(), embs2.min())

0.55150765 -0.5166339
11.068921 -10.268374


In [26]:
embs2_norm = l2_normalize(embs2)

In [61]:
print(embs2_norm.max(), embs2_norm.min())
print(embs2_norm.shape)

0.5515077 -0.516634
(7, 768)


In [35]:
embs2_norm = l2_normalize(embs2_norm)

In [59]:
t=embs2_norm[0]
sum(embs2_norm[0])

0.5019712232710845

the length = 1 , not the sum  !!!!   

In [67]:
print(np.linalg.norm(embs2_norm[0]))
print(np.linalg.norm(embs2_norm[1]))
print(np.linalg.norm(embs2_norm[2]))
print(np.linalg.norm(embs2_norm[3]))
print(np.linalg.norm(embs2_norm[4]))
print(np.linalg.norm(embs2_norm[5]))
print(np.linalg.norm(embs2_norm[6]))

0.99999994
1.0
1.0000001
1.0000001
1.0
1.0
1.0


-------------------

putting it all together

------------------

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

## or this in case of running from outer folder
# sys.path.append(str(Path("../src").resolve()))


from uiflow.io.load import read_meta_tsv, read_embeddings_npy, attach_screen_key, align_by_key, l2_normalize
from uiflow.repr.fuse import concat_fuse
from uiflow.analysis.clustering import kmeans_cluster




In [2]:
DATA_PATH = Path("../../processed_data").resolve()

In [3]:
clip_meta = attach_screen_key(read_meta_tsv(DATA_PATH/"clip_embeddings_with_ids.tsv"))
clip_emb = l2_normalize(read_embeddings_npy(DATA_PATH/"clip_embeddings.npy"))

txt_meta = attach_screen_key(read_meta_tsv(DATA_PATH/"sbert_text_meta.tsv"))
txt_emb = l2_normalize(read_embeddings_npy(DATA_PATH/"sbert_text_embeddings.npy"))
meta, v, t = align_by_key(clip_meta, clip_emb, txt_meta, txt_emb)
fused = concat_fuse(v, t, alpha=1.0, beta=1.0)

cid = kmeans_cluster(fused, k=80)

KeyError: "meta: missing columns ['screen_id']. Have: ['app_id', 'trace_id', 'image_path', 'e0', 'e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'e7', 'e8', 'e9', 'e10', 'e11', 'e12', 'e13', 'e14', 'e15', 'e16', 'e17', 'e18', 'e19', 'e20', 'e21', 'e22', 'e23', 'e24', 'e25', 'e26', 'e27', 'e28', 'e29', 'e30', 'e31', 'e32', 'e33', 'e34', 'e35', 'e36', 'e37', 'e38', 'e39', 'e40', 'e41', 'e42', 'e43', 'e44', 'e45', 'e46', 'e47', 'e48', 'e49', 'e50', 'e51', 'e52', 'e53', 'e54', 'e55', 'e56', 'e57', 'e58', 'e59', 'e60', 'e61', 'e62', 'e63', 'e64', 'e65', 'e66', 'e67', 'e68', 'e69', 'e70', 'e71', 'e72', 'e73', 'e74', 'e75', 'e76', 'e77', 'e78', 'e79', 'e80', 'e81', 'e82', 'e83', 'e84', 'e85', 'e86', 'e87', 'e88', 'e89', 'e90', 'e91', 'e92', 'e93', 'e94', 'e95', 'e96', 'e97', 'e98', 'e99', 'e100', 'e101', 'e102', 'e103', 'e104', 'e105', 'e106', 'e107', 'e108', 'e109', 'e110', 'e111', 'e112', 'e113', 'e114', 'e115', 'e116', 'e117', 'e118', 'e119', 'e120', 'e121', 'e122', 'e123', 'e124', 'e125', 'e126', 'e1